#  Data Cleaning

In [1]:
import sys
from pathlib import Path

import pandas as pd

BASE = Path.cwd()
if BASE.name == "notebooks":
    BASE = BASE.parent
sys.path.insert(0, str(BASE / "src"))

from cleaning import (clean_orders, clean_products, validate_emails,
                      check_referential_integrity, clean_order_items)

RAW = BASE / "data" / "raw"
CLEANED = BASE / "data" / "cleaned"
CLEANED.mkdir(parents=True, exist_ok=True)

orders_raw = pd.read_csv(RAW / "orders.csv", dtype={"customer_id": "string"}, keep_default_na=False)
items_raw = pd.read_csv(RAW / "order_items.csv")
products_raw = pd.read_csv(RAW / "products.csv")
customers_raw = pd.read_csv(RAW / "customers.csv")

print(f"orders: {len(orders_raw)}, order_items: {len(items_raw)}, "
      f"products: {len(products_raw)}, customers: {len(customers_raw)}")

orders: 2000, order_items: 5000, products: 500, customers: 800


## 1. clean_orders() - dates and NULL customer_ids

In [2]:
orders_clean, order_issues = clean_orders(orders_raw)

print(f"NULL/empty customer_id: {order_issues['null_customer_ids']['count']} orders (kept as guest orders)")
print(f"Wrong date format fixed: {order_issues['wrong_date_format']['count']} orders")
print(f"Unparseable dates: {order_issues['unparseable_dates']['count']} orders (dropped)")
print(f"Future-dated orders: {order_issues['future_dates']['count']} orders (dropped) "
      f"{order_issues['future_dates']['order_ids']}")

assert orders_clean["order_date"].str.match(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$").all()
print("\nAll order_date values now in YYYY-MM-DD HH:MM:SS format")
orders_clean.head()

NULL/empty customer_id: 100 orders (kept as guest orders)
Wrong date format fixed: 60 orders
Unparseable dates: 0 orders (dropped)
Future-dated orders: 3 orders (dropped) ['O00039', 'O00194', 'O01438']

All order_date values now in YYYY-MM-DD HH:MM:SS format


,order_id,customer_id,order_date,status,region_code
0,O00001,C0353,2026-04-29 16:15:01,DELIVERED,WEST
1,O00002,C0277,2025-10-20 16:00:39,PLACED,CENTRAL
2,O00003,C0225,2026-03-13 16:33:50,CANCELLED,EAST
3,O00004,C0060,2026-02-15 08:50:03,DELIVERED,CENTRAL
4,O00005,C0239,2025-08-31 23:37:19,PLACED,SOUTH


## 2. clean_products() - normalize names

In [3]:
products_clean, product_issues = clean_products(products_raw)

n = product_issues["messy_product_names"]["count"]
print(f"Product names that are normalized count: {n}")

fixed_ids = product_issues["messy_product_names"]["product_ids"][:5]
comparison = pd.DataFrame({
    "before": products_raw.set_index("product_id").loc[fixed_ids, "product_name"],
    "after": products_clean.set_index("product_id").loc[fixed_ids, "product_name"],
})
comparison

Product names that are normalized count: 30


,before,after
product_id,,
P0010,krueger essential encyclopedia,Krueger Essential Encyclopedia
P0032,Sanchez Pro Bookshelf,Sanchez Pro Bookshelf
P0045,tucker pro organizer,Tucker Pro Organizer
P0071,jones compact bedsheet,Jones Compact Bedsheet
P0103,Atkinson Ultra Jacket,Atkinson Ultra Jacket


## 3. validate_emails() - invalid email report

In [4]:
invalid_ids = validate_emails(customers_raw)
print(f"Customers with invalid emails are: {len(invalid_ids)}")
customers_raw[customers_raw["customer_id"].isin(invalid_ids)][["customer_id", "customer_name", "email"]]

Customers with invalid emails are: 16


,customer_id,customer_name,email
152,C0153,Joshua Holland,joshua.holland@
289,C0290,Rachel Young,rachel.younggmail.com
305,C0306,Alexis Robertson,alexis.robertsongmail.com
340,C0341,Angelica Parker,angelica.parkerhotmail.com
353,C0354,Benjamin Brown,benjamin.brown@
457,C0458,Jennifer Kirby,jennifer.kirbygmail.com
495,C0496,Rachael Nguyen,rachael.nguyenyahoo.com
501,C0502,Tammy Kane,tammy.kaneyahoo.com
551,C0552,Sara Smith,sara.smith@
556,C0557,Michele Smith,michele.smith@


## 4. check_referential_integrity() and clean_order_items()

In [5]:
integrity = check_referential_integrity(items_raw, orders_clean, products_raw)
print(f"order_items referencing non-existent orders : {integrity['orphan_order_items']['count']}")
print(f"  missing order_ids: {integrity['orphan_order_items']['missing_order_ids']}")
print(f"order_items referencing non-existent products: {integrity['orphan_product_items']['count']}")

items_clean, item_issues = clean_order_items(items_raw, orders_clean)
print(f"\nAfter cleaning: {len(items_clean)} rows (dropped {len(items_raw) - len(items_clean)} orphans)")
print(f"discount_percent out of range (clamped to 0-100): {item_issues['discount_out_of_range']['count']}")
print(f"zero-quantity rows (kept, flagged): {item_issues['zero_quantity']['count']}")
print(f"negative-quantity rows (kept - returns): {item_issues['negative_quantity_returns']['count']}")

assert items_clean["discount_percent"].between(0, 100).all()
assert items_clean["order_id"].isin(orders_clean["order_id"]).all()
print("\nReferential integrity and discount range both are verified on cleaned data")

order_items referencing non-existent orders : 23
  missing order_ids: ['O00039', 'O00194', 'O01438', 'O99900', 'O99901', 'O99902', 'O99903', 'O99904', 'O99905', 'O99906', 'O99907', 'O99908', 'O99909', 'O99910', 'O99911', 'O99912', 'O99913', 'O99914']
order_items referencing non-existent products: 0

After cleaning: 4977 rows (dropped 23 orphans)
discount_percent out of range (clamped to 0-100): 5
zero-quantity rows (kept, flagged): 5
negative-quantity rows (kept - returns): 150

Referential integrity and discount range both are verified on cleaned data


## 5. Save cleaned CSVs and write the data quality report

In [6]:
orders_clean.to_csv(CLEANED / "orders.csv", index=False)
items_clean.to_csv(CLEANED / "order_items.csv", index=False)
products_clean.to_csv(CLEANED / "products.csv", index=False)
customers_raw.to_csv(CLEANED / "customers.csv", index=False)

report_lines = [
    "=" * 70,
    "DATA QUALITY REPORT - E-Commerce Order Analytics",
    "=" * 70,
    "",
    "ORDERS",
    f"Rows in / out: {len(orders_raw)} / {len(orders_clean)}",
    f"NULL or empty customer_id: {order_issues['null_customer_ids']['count']} (kept as guest orders, customer_id set to NULL)",
    f"Dates in wrong DD-MM-YYYY format : {order_issues['wrong_date_format']['count']} (converted to YYYY-MM-DD HH:MM:SS)",
    f"Unparseable dates: {order_issues['unparseable_dates']['count']} (dropped)",
    f"Future-dated orders: {order_issues['future_dates']['count']} (dropped) -> {order_issues['future_dates']['order_ids']}",
    "",
    "ORDER ITEMS",
    f"Rows in / out: {len(items_raw)} / {len(items_clean)}",
    f"Referencing non-existent orders: {item_issues['orphan_order_items']['count']} (dropped)",
    f"missing order_ids: {item_issues['orphan_order_items']['missing_order_ids']}",
    f"discount_percent out of 0-100: {item_issues['discount_out_of_range']['count']} (clamped) -> {item_issues['discount_out_of_range']['item_ids']}",
    f"Zero quantity: {item_issues['zero_quantity']['count']} (kept, contribute no revenue) -> {item_issues['zero_quantity']['item_ids']}",
    f"Negative quantity(returns): {item_issues['negative_quantity_returns']['count']} (kept - legitimate returns)",
    "",
    "PRODUCTS",
    f"Rows: {len(products_clean)}",
    f"Names normalized (spaces/case): {product_issues['messy_product_names']['count']}",
    "",
    "CUSTOMERS",
    f"Rows: {len(customers_raw)}",
    f"Invalid emails: {len(invalid_ids)} -> {invalid_ids}",
    "",
    "=" * 70,
]
report = "\n".join(report_lines)
(CLEANED / "data_quality_report.txt").write_text(report, encoding="utf-8")
print(report)

DATA QUALITY REPORT - E-Commerce Order Analytics

ORDERS
Rows in / out: 2000 / 1997
NULL or empty customer_id: 100 (kept as guest orders, customer_id set to NULL)
Dates in wrong DD-MM-YYYY format : 60 (converted to YYYY-MM-DD HH:MM:SS)
Unparseable dates: 0 (dropped)
Future-dated orders: 3 (dropped) -> ['O00039', 'O00194', 'O01438']

ORDER ITEMS
Rows in / out: 5000 / 4977
Referencing non-existent orders: 23 (dropped)
missing order_ids: ['O00039', 'O00194', 'O01438', 'O99900', 'O99901', 'O99902', 'O99903', 'O99904', 'O99905', 'O99906', 'O99907', 'O99908', 'O99909', 'O99910', 'O99911', 'O99912', 'O99913', 'O99914']
discount_percent out of 0-100: 5 (clamped) -> ['I00987', 'I01430', 'I01515', 'I03494', 'I03965']
Zero quantity: 5 (kept, contribute no revenue) -> ['I00191', 'I01241', 'I01837', 'I03354', 'I03366']
Negative quantity(returns): 150 (kept - legitimate returns)

PRODUCTS
Rows: 500
Names normalized (spaces/case): 30

CUSTOMERS
Rows: 800
Invalid emails: 16 -> ['C0153', 'C0290', 'C030